In [48]:
import torch
from jaxtyping import Float, Int
from torch import Tensor
from llm.nn import softmax
from einops import rearrange

In [19]:
def run_cross_entropy(
    inputs: Float[Tensor, " batch_size vocab_size"], targets: Int[Tensor, " batch_size"]
) -> Float[Tensor, ""]:
    """Given a tensor of inputs and targets, compute the average cross-entropy
    loss across examples.

    Args:
        inputs (Float[Tensor, "batch_size vocab_size"]): inputs[i][j] is the
            unnormalized logit of jth class for the ith example.
        targets (Int[Tensor, "batch_size"]): Tensor of shape (batch_size,) with the index of the correct class.
            Each value must be between 0 and `num_classes - 1`.

    Returns:
        Float[Tensor, ""]: The average cross-entropy loss across examples.
    """
    pass
    


In [5]:
batch_size = 10
vocab_size = 1024

inputs = torch.randn(batch_size, vocab_size)

In [6]:
inputs

tensor([[-0.0479, -0.8070, -0.4829,  ..., -1.2783, -0.8695,  1.1840],
        [ 0.4163, -1.2989, -1.0470,  ...,  0.1809,  0.7147, -0.9860],
        [-0.3217,  0.0365,  1.2506,  ..., -0.7164, -0.3164, -0.4974],
        ...,
        [-0.5686,  0.0846,  1.3836,  ...,  0.3468,  0.0414,  1.5134],
        [-0.4687, -0.1684,  0.3070,  ..., -0.0492,  0.5271,  0.2029],
        [ 1.3190, -0.5801, -0.4732,  ..., -0.2927, -1.0156,  0.8672]])

In [9]:
targets = torch.randint(low=0, high=vocab_size, size=(batch_size, ))

In [10]:
targets.shape

torch.Size([10])

In [17]:
targets

tensor([581, 422, 247, 874, 465, 900, 193, 504, 762, 458])

In [13]:
p = softmax(inputs)

In [14]:
p

tensor([[0.0006, 0.0003, 0.0004,  ..., 0.0002, 0.0003, 0.0020],
        [0.0009, 0.0002, 0.0002,  ..., 0.0007, 0.0012, 0.0002],
        [0.0004, 0.0006, 0.0021,  ..., 0.0003, 0.0004, 0.0004],
        ...,
        [0.0004, 0.0007, 0.0025,  ..., 0.0009, 0.0006, 0.0028],
        [0.0004, 0.0005, 0.0009,  ..., 0.0006, 0.0011, 0.0008],
        [0.0022, 0.0003, 0.0004,  ..., 0.0004, 0.0002, 0.0014]])

In [15]:
torch.sum(p, dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000])

In [36]:
p.shape

torch.Size([10, 1024])

In [ ]:
p[0]
torch.index_select(p, dim=-1, index=targets)

In [74]:
inputs = torch.tensor(
    [
        [
            [0.1088, 0.1060, 0.6683, 0.5131, 0.0645],
            [0.4538, 0.6852, 0.2520, 0.3792, 0.2675],
            [0.4578, 0.3357, 0.6384, 0.0481, 0.5612],
            [0.9639, 0.8864, 0.1585, 0.3038, 0.0350],
        ],
        [
            [0.3356, 0.9013, 0.7052, 0.8294, 0.8334],
            [0.6333, 0.4434, 0.1428, 0.5739, 0.3810],
            [0.9476, 0.5917, 0.7037, 0.2987, 0.6208],
            [0.8541, 0.1803, 0.2054, 0.4775, 0.8199],
        ],
    ]
)
targets = torch.tensor([[1, 0, 2, 2], [4, 1, 4, 0]])


In [75]:
inputs = inputs.view(-1, inputs.size(-1))
targets = targets.view(-1)


In [76]:
inputs.shape, targets.shape

(torch.Size([8, 5]), torch.Size([8]))

In [77]:
inputs, targets

(tensor([[0.1088, 0.1060, 0.6683, 0.5131, 0.0645],
         [0.4538, 0.6852, 0.2520, 0.3792, 0.2675],
         [0.4578, 0.3357, 0.6384, 0.0481, 0.5612],
         [0.9639, 0.8864, 0.1585, 0.3038, 0.0350],
         [0.3356, 0.9013, 0.7052, 0.8294, 0.8334],
         [0.6333, 0.4434, 0.1428, 0.5739, 0.3810],
         [0.9476, 0.5917, 0.7037, 0.2987, 0.6208],
         [0.8541, 0.1803, 0.2054, 0.4775, 0.8199]]),
 tensor([1, 0, 2, 2, 4, 1, 4, 0]))

In [78]:
targets = rearrange(targets, "batch_size -> batch_size 1")

In [88]:
def logsoftmax(x: torch.Tensor) -> torch.Tensor:
    return x - x.exp().sum(-1).log()

probs = softmax(inputs, dim=-1)

raw = inputs.exp().sum(-1).log()
print(raw.shape, inputs.shape)
raw_unsq = raw.unsqueeze(-1)
print(raw_unsq.shape, raw_unsq)

log_probs = inputs - raw_unsq
log_probs

torch.Size([8]) torch.Size([8, 5])
torch.Size([8, 1]) tensor([[1.9337],
        [2.0298],
        [2.0380],
        [2.1530],
        [2.3494],
        [2.0585],
        [2.2635],
        [2.1584]])


tensor([[-1.8249, -1.8277, -1.2654, -1.4206, -1.8692],
        [-1.5760, -1.3446, -1.7778, -1.6506, -1.7623],
        [-1.5802, -1.7023, -1.3996, -1.9899, -1.4768],
        [-1.1891, -1.2666, -1.9945, -1.8492, -2.1180],
        [-2.0138, -1.4481, -1.6442, -1.5200, -1.5160],
        [-1.4252, -1.6151, -1.9157, -1.4846, -1.6775],
        [-1.3159, -1.6718, -1.5598, -1.9648, -1.6427],
        [-1.3043, -1.9781, -1.9530, -1.6809, -1.3385]])

In [62]:
probs = torch.gather(inputs, -1, targets)

In [63]:
probs

tensor([[0.1060],
        [0.4538],
        [0.6384],
        [0.1585],
        [0.8334],
        [0.4434],
        [0.6208],
        [0.8541]])

In [66]:
probs = softmax(inputs, dim=-1)

In [67]:
probs

tensor([[0.1612, 0.1608, 0.2821, 0.2416, 0.1543],
        [0.2068, 0.2606, 0.1690, 0.1919, 0.1716],
        [0.2059, 0.1823, 0.2467, 0.1367, 0.2284],
        [0.3045, 0.2818, 0.1361, 0.1574, 0.1203],
        [0.1335, 0.2350, 0.1932, 0.2187, 0.2196],
        [0.2405, 0.1989, 0.1472, 0.2266, 0.1868],
        [0.2682, 0.1879, 0.2102, 0.1402, 0.1935],
        [0.2714, 0.1383, 0.1418, 0.1862, 0.2622]])

In [68]:
probs.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [69]:
probs_i = torch.gather(probs, -1, targets)

In [70]:
probs_i

tensor([[0.1608],
        [0.2068],
        [0.2467],
        [0.1361],
        [0.2196],
        [0.1989],
        [0.1935],
        [0.2714]])

In [71]:
torch.mean(-torch.log(probs_i))

tensor(1.6095)

logsoftmax = log(e^xi/sum(e